# Justification annotation — pilot review

Checking the DeepSeek annotations of LLM vote justifications against the scheme in
`src/prompts/justification_annotation.txt`.

**How to use this notebook**

1. Run sections 1–3 to see whether the annotator followed the contract, what the
   full corpus run will cost, and what it found.
2. Section 4 is the reviewer. Work through the justifications; every click saves
   immediately, so you can stop and come back.
3. Sections 5–6 score your review and export the disagreements.

Your verdicts go to `pilot_review_verdicts.csv` and `pilot_review_missed.csv`
alongside the annotations. Re-running this notebook never overwrites them.


In [ ]:
# ============================================================
# Setup
# ============================================================

from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt


def find_repo_root(start=None, repo_name="masters_thesis_sdg"):
    current = (start or Path.cwd()).resolve()
    while True:
        if current.name == repo_name:
            return current
        if current.parent == current:
            raise FileNotFoundError(f"Could not find repo root {repo_name!r} above {Path.cwd()}")
        current = current.parent


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.pt_annotation.justification_schema import CATEGORY_COLORS
from src.pt_annotation.justification_review_tools import (
    load_pilot, annotations_long, sentences_long, usage_frame,
    VerdictStore, score,
)
from src.pt_annotation.justification_review_widget import ReviewApp

PILOT_DIR = REPO_ROOT / "results" / "justification_annotation" / "pilot_v1"

# The schema is detected from the saved metadata; for this folder it is v1.
records, sample, schema = load_pilot(PILOT_DIR)
CATEGORIES = list(schema.categories)
annotations = annotations_long(records)
sentences = sentences_long(records)
usage = usage_frame(records)

# The whole corpus this pilot is standing in for.
CORPUS_SIZE = 2287

print(f"{len(records)} justifications, {len(sentences)} sentences, {len(annotations)} annotations")

## 1. Did the annotator follow the contract?

This has to hold before anything else is worth reading. The runner already
validated every response; this re-derives the checks from the saved output so
the notebook stands on its own.

The load-bearing one is **span exactness** — `evidence_span` must be a verbatim
substring of its sentence. Paraphrased spans cannot be aligned back to the text,
which would make every span-level measure unusable and break the highlighting below.

In [ ]:
# ============================================================
# 1. Contract compliance
# ============================================================

flagged = [r for r in records if r["metadata"].get("validation_flags")]
errored = [r for r in records if "error" in r["metadata"]]
truncated = usage[usage["finish_reason"].ne("stop")]

spans_missing = annotations[
    [s not in t for s, t in zip(annotations["evidence_span"].fillna(""), annotations["text"])]
]

expected_sentences = sum(r["metadata"]["n_sentences"] for r in records)

checks = pd.DataFrame([
    ("justifications returned",       f"{len(records)}/{len(sample)}"),
    ("API or parse errors",           len(errored)),
    ("validation flags raised",       len(flagged)),
    ("responses hit the token cap",   len(truncated)),
    ("evidence_span not verbatim",    len(spans_missing)),
    ("categories outside the scheme", int((~annotations["category"].isin(CATEGORIES)).sum())),
    ("sentences dropped or invented", int(len(sentences) - expected_sentences)),
], columns=["check", "result"])

display(checks.style.hide(axis="index"))

if flagged:
    print("\nFlags raised:")
    for record in flagged:
        print(" ", record["metadata"]["justification_id"])
        for flag in record["metadata"]["validation_flags"]:
            print("   -", flag)
else:
    print("\nNo flags. Every response preserved the vote, the sentence ids, the sentence")
    print("text, the category and use vocabularies, and quoted spans verbatim.")

## 2. What will the full corpus run cost?

The pilot is the only measurement available for projecting the full run. Thinking
tokens dominate: they are billed as output and count toward `max_tokens`.

Set `PRICE_INPUT` / `PRICE_OUTPUT` to your actual DeepSeek rates (per million
tokens) to turn the token projection into a number.

In [ ]:
# ============================================================
# 2. Scale and cost projection
# ============================================================

PRICE_INPUT = None    # e.g. 0.28  (per 1M input tokens, in your currency)
PRICE_OUTPUT = None   # e.g. 0.42  (per 1M output tokens)

per_call = usage[["prompt_tokens", "completion_tokens", "reasoning_tokens", "total_tokens"]]

summary = pd.DataFrame({
    "mean": per_call.mean().round(0),
    "max": per_call.max(),
    "pilot total": per_call.sum(),
    f"projected ({CORPUS_SIZE})": (per_call.mean() * CORPUS_SIZE).round(0),
}).astype("Int64")

display(summary)

reasoning_share = 100 * usage["reasoning_tokens"].sum() / usage["completion_tokens"].sum()
headroom = 100 * usage["completion_tokens"].max() / 32768

print(f"Thinking is {reasoning_share:.0f}% of all output tokens.")
print(f"Largest response used {headroom:.0f}% of the 32,768 max_tokens cap "
      f"({usage['completion_tokens'].max():,} tokens) - nothing was truncated.")

if PRICE_INPUT is not None and PRICE_OUTPUT is not None:
    projected = (
        per_call["prompt_tokens"].mean() * CORPUS_SIZE / 1e6 * PRICE_INPUT
        + per_call["completion_tokens"].mean() * CORPUS_SIZE / 1e6 * PRICE_OUTPUT
    )
    pilot_cost = (
        per_call["prompt_tokens"].sum() / 1e6 * PRICE_INPUT
        + per_call["completion_tokens"].sum() / 1e6 * PRICE_OUTPUT
    )
    print(f"\nPilot cost {pilot_cost:,.2f}; full corpus projected {projected:,.2f}.")
else:
    print("\nSet PRICE_INPUT and PRICE_OUTPUT above for a cost estimate.")

## 3. What did it find?

Read this before reviewing — knowing the distribution tells you which boundaries
to scrutinise. A category that barely fires has not been exercised by the pilot,
so the review says nothing about whether the prompt handles it.

In [ ]:
# ============================================================
# 3. Label distribution
# ============================================================

n_sentences = len(sentences)
n_labelled = int(sentences["is_labelled"].sum())

print(f"labelled sentences   : {n_labelled}/{n_sentences} ({100*n_labelled/n_sentences:.1f}%)")
print(f"multi-label sentences: {int((sentences['n_annotations'] > 1).sum())} "
      f"({100*(sentences['n_annotations'] > 1).mean():.1f}%)")
print(f"rule_mentioned       : {int(sentences['rule_mentioned'].sum())} "
      f"({100*sentences['rule_mentioned'].mean():.1f}%)")

distribution = (
    annotations.groupby("category").size().reindex(CATEGORIES).fillna(0).astype(int)
    .to_frame("n_annotations")
)
distribution["pct"] = (100 * distribution["n_annotations"] / len(annotations)).round(1)
distribution["exercised"] = distribution["n_annotations"] >= 5
display(distribution)

use_by_category = (
    annotations.groupby(["category", "use"]).size().unstack(fill_value=0)
    .reindex(index=CATEGORIES, columns=["used", "discounted", "mentioned"], fill_value=0)
)
display(use_by_category)

In [ ]:
# ============================================================
# 3b. Two views: what fires, and how it is used
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

order = distribution.sort_values("n_annotations", ascending=True)
axes[0].barh(
    order.index, order["n_annotations"],
    color=[CATEGORY_COLORS[c] for c in order.index], edgecolor="#999",
)
axes[0].set_title("Annotations per category")
axes[0].set_xlabel("count")

shares = use_by_category.div(use_by_category.sum(axis=1).replace(0, 1), axis=0) * 100
bottom = pd.Series(0.0, index=shares.index)
for use_value, colour in [("used", "#4c78a8"), ("discounted", "#e45756"), ("mentioned", "#b0b0b0")]:
    axes[1].bar(shares.index, shares[use_value], bottom=bottom, label=use_value, color=colour)
    bottom += shares[use_value]
axes[1].set_title("How evidence is used, within category")
axes[1].set_ylabel("% of annotations")
axes[1].legend(fontsize=8)
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 3c. Unlabelled sentences and Other
#
# Unlabelled is a deliberate outcome, not a gap: the prompt says not to force an
# annotation onto every sentence. These are the ones to sanity-check, because a
# silently missed label looks exactly like correct restraint.
# ============================================================

unlabelled = sentences[~sentences["is_labelled"]]
print(f"{len(unlabelled)} unlabelled sentences\n")
for row in unlabelled.itertuples():
    print(f"  [{row.model}] {row.text}")

other = annotations[annotations["category"].eq("Other")]
if len(other):
    print(f"\n{len(other)} Other annotations - check whether they name one recurring pattern:")
    for row in other.itertuples():
        print(f"  - {row.other_description}")
        print(f"      \"{row.evidence_span}\"")

## 4. Review

One justification at a time. Evidence spans are highlighted in the sentence and
colour-coded by category.

For each annotation set a **verdict**:

| verdict | meaning |
|---|---|
| `ok` | the annotation is right |
| `wrong category` | right that something is here, wrong label — set the correction |
| `wrong use` | right label, wrong `used`/`discounted`/`mentioned` |
| `wrong span` | right label, the quoted span is not the right evidence |
| `spurious` | nothing should have been annotated here |

**"missed a label?"** on each sentence catches the opposite failure — something
the annotator should have labelled and didn't. That's the one this format can't
otherwise express, since there's no row to disagree with.

**Mark all ok** accepts every not-yet-judged annotation on the current
justification. It never overwrites a verdict you already set.

In [ ]:
# ============================================================
# 4. Reviewer
# ============================================================

store = VerdictStore(
    PILOT_DIR / "pilot_review_verdicts.csv",
    PILOT_DIR / "pilot_review_missed.csv",
)

app = ReviewApp(records, store, schema, sample)
app.display()

## 5. How much did you agree with?

Re-run this after reviewing — it reads the saved verdicts, so it reflects
progress even if you stopped part-way.

This is an **error rate against your adjudication**, not a chance-corrected
agreement coefficient. You reviewed the model's labels rather than coding blind,
so the two label sets aren't independent and κ/α wouldn't mean what they
normally mean.

In [ ]:
# ============================================================
# 5. Score
# ============================================================

store = VerdictStore(
    PILOT_DIR / "pilot_review_verdicts.csv",
    PILOT_DIR / "pilot_review_missed.csv",
)

per_category, summary = score(store.verdicts, annotations, schema)

if not summary:
    print("No verdicts recorded yet - work through section 4 first.")
else:
    print(f"reviewed    : {summary['n_reviewed']}/{summary['n_total']} annotations")
    print(f"accept rate : {summary['accept_rate']:.1f}%")
    print(f"verdicts    : {summary['verdict_counts']}")
    print(f"missed      : {len(store.missed)} sentences flagged as missing a label")
    print()
    display(per_category)

    reviewed = per_category[per_category["n_reviewed"] > 0]
    if len(reviewed):
        fig, ax = plt.subplots(figsize=(7, 3.5))
        order = reviewed.sort_values("accept_rate")
        ax.barh(order.index, order["accept_rate"],
                color=[CATEGORY_COLORS[c] for c in order.index], edgecolor="#999")
        ax.set_xlim(0, 100)
        ax.set_xlabel("% of annotations accepted")
        ax.set_title("Where the annotator and you disagree")
        for i, (n, rate) in enumerate(zip(order["n_reviewed"], order["accept_rate"])):
            ax.text(2, i, f"n={n}", va="center", fontsize=8, color="#333")
        plt.tight_layout()
        plt.show()

## 6. Export the disagreements

Everything you marked as wrong, in one table — this is what drives prompt
revisions before the full run. A category with a low accept rate points at a
boundary the prompt states badly; scattered one-off errors point at nothing.

In [ ]:
# ============================================================
# 6. Findings
# ============================================================

disagreements = store.verdicts[
    store.verdicts["verdict"].ne("") & store.verdicts["verdict"].ne("ok")
].copy()

if disagreements.empty:
    print("No disagreements recorded.")
else:
    display(disagreements)

    confusion = (
        disagreements[disagreements["corrected_category"].ne("")]
        .groupby(["category", "corrected_category"]).size()
        .rename("n").reset_index()
        .sort_values("n", ascending=False)
    )
    if len(confusion):
        print("\nWhich boundaries are being crossed (annotator said -> you said):")
        display(confusion)

if not store.missed.empty:
    print("\nSentences you flagged as missing a label:")
    display(store.missed)

out_path = PILOT_DIR / "tables" / "08_review_findings.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
disagreements.to_csv(out_path, index=False, encoding="utf-8")
print(f"\nWritten to {out_path.relative_to(REPO_ROOT)}")